# 1. Datenaufbereitung

Lädt die Rohdaten, trennt Fragen mit fester Antwort von Freitext-Fragen, bereinigt die Daten und speichert eine verarbeitete Version für die weiteren Notebooks.

In [10]:
import sys
from pathlib import Path

import pandas as pd

sys.path.append(str(Path.cwd().parent))
from src.utils import load_survey_data

RAW_PATH = Path("../data/raw/results-survey535447.csv")
PROCESSED_PATH = Path("../data/processed/survey_clean.csv")

In [11]:
df = load_survey_data(RAW_PATH)
print(df.shape)
df["Last page"].value_counts(dropna=False)

(44, 57)


Last page
4.0    26
NaN     7
2.0     5
3.0     5
1.0     1
Name: count, dtype: int64

## Nur vollständig ausgefüllte Antworten behalten

LimeSurvey setzt `Date submitted` nur, wenn die Umfrage komplett abgeschlossen wurde (hier identisch mit `Last page == 4`). Abgebrochene Antworten (`Date submitted` leer) werden für die Analyse ausgeschlossen.

In [12]:
df_completed = df[df["Date submitted"].notna()].reset_index(drop=True)
print(f"{len(df_completed)} von {len(df)} Antworten wurden vollständig abgeschlossen.")
df_completed.head()

26 von 44 Antworten wurden vollständig abgeschlossen.


,Response ID,Date submitted,Last page,Start language,Seed,Which of the following best describes the highest level of formal education that you've completed?,"Including any education, how many years have you been coding in total? Please round to the nearest whole number, excluding any decimal points.",How did you learn to code in the past year? [Technical documentation (is generated for/by the tool or system],"How did you learn to code in the past year? [Other online resources (e.g. standard search, forum, online community]",How did you learn to code in the past year? [Stack Overflow or Stack Exchange],...,Which projects or operational constraints have the greatest influence on software development decisions in your field of work? Please briefly explain why the selected constraints have a particularly strong influence on software development decisions in your field of work. [Legacy systems][Comment],Which projects or operational constraints have the greatest influence on software development decisions in your field of work? Please briefly explain why the selected constraints have a particularly strong influence on software development decisions in your field of work. [Organizational constraints],Which projects or operational constraints have the greatest influence on software development decisions in your field of work? Please briefly explain why the selected constraints have a particularly strong influence on software development decisions in your field of work. [Organizational constraints][Comment],Which projects or operational constraints have the greatest influence on software development decisions in your field of work? Please briefly explain why the selected constraints have a particularly strong influence on software development decisions in your field of work. [I don't know],Which projects or operational constraints have the greatest influence on software development decisions in your field of work? Please briefly explain why the selected constraints have a particularly strong influence on software development decisions in your field of work. [I don't know][Comment],To what extent can development teams in your field of work choose or adapt the software development process they use?,Which factors have the greatest influence on the choice or adaption of the software development process?,"In your opinion, will software engineering in the space sector and software engineering in other domains become more similar or more different in the future? Please briefly explain the reason for your answer.","In your opinion, will software engineering in the space sector and software engineering in other domains become more similar or more different in the future? Please briefly explain the reason for your answer. [Comment]",Is there anything else you would like to share that is relevant to this topic?
0,5,1980-01-01 00:00:00,4.0,en,238307838,"Master’s degree (M.A., M.S., M.Eng., MBA, etc.)",6 to 10 years,No,Yes,Yes,...,NaN,No,NaN,No,NaN,High flexibility,NaN,NaN,NaN,NaN
1,9,1980-01-01 00:00:00,4.0,en,193893040,"Master’s degree (M.A., M.S., M.Eng., MBA, etc.)",16 to 20 years,Yes,Yes,Yes,...,This goes together with the long mission lifet...,No,NaN,No,NaN,Moderate flexibility,- ECSS standards\n- Previous experience of the...,Become somewhat more similar,More standardized platforms/interfaces,NaN
2,10,1980-01-01 00:00:00,4.0,en,1433496783,"Master’s degree (M.A., M.S., M.Eng., MBA, etc.)",11 to 15 years,Yes,Yes,Yes,...,NaN,No,NaN,No,NaN,High flexibility,NaN,Remain approximately as they are today,NaN,NaN
3,11,1980-01-01 00:00:00,4.0,en,1373684769,"Professional degree (JD, MD, Ph.D, Ed.D, etc.)",41 to 50 years,Yes,Yes,Yes,...,NaN,Yes,Fullfilment of ECSS requirements,No,NaN,Limited flexibility,Acceptance by the customers (ESA or NASA stand...,Become much more different,There is a big difference between the domains....,NaN
4,12,1980-01-01 00:00:00,4.0,en,1914825945,"Master’s degree (M.A., M.S., M.Eng., MBA, etc.)",6 to 10 years,Yes,No,Yes,...,Na

## Spalten mit fester Antwort vs. Freitext

Die Spaltennamen sind sehr lang (LimeSurvey-Exportformat) und enthalten teils geschützte Leerzeichen. Deshalb werden die Fragegruppen hier über ihre **Position** in der (stabilen) Spaltenreihenfolge referenziert statt über den vollen Text.

In [13]:
cols = df_completed.columns

# Einzel-Fragen mit fester Antwort (Single-Choice / ordinal)
education = cols[5]
years_coding_total = cols[6]
learned_ai_at_work = cols[19]
job_role = cols[20]
years_experience = cols[21]
languages_used = cols[22]
uses_ai_tools = cols[23]
ai_stance = cols[24]
ai_threat = cols[25]
process_flexibility = cols[52]
similar_or_different = cols[54]

# Multiple-Choice-Gruppen: je Option eine eigene Yes/No-Spalte
learn_to_code_columns = cols[7:19].tolist()  # 12 Lernquellen
nonfunctional_req_columns = cols[26:34].tolist()  # 8 nicht-funktionale Anforderungen
constraint_columns = cols[34:52:2].tolist()  # 9 operative Einschränkungen (Yes/No)
constraint_comment_columns = cols[35:52:2].tolist()  # zugehörige Freitext-Begründungen

# Freitext-Fragen
process_factors_text = cols[53]
similar_or_different_comment = cols[55]
anything_else_text = cols[56]

print("Single-Choice-Spalten:", [education, years_coding_total, learned_ai_at_work, job_role,
                                  years_experience, languages_used, uses_ai_tools, ai_stance,
                                  ai_threat, process_flexibility, similar_or_different])
print("\nLerntquellen (Multi-Select):", learn_to_code_columns)
print("\nNicht-funktionale Anforderungen (Multi-Select):", nonfunctional_req_columns)
print("\nOperative Einschränkungen (Multi-Select):", constraint_columns)
print("\nFreitext-Spalten:", [process_factors_text, similar_or_different_comment, anything_else_text])

Single-Choice-Spalten: ["Which of the following best describes the highest level of formal education that you've completed?", 'Including any education, how many years have you been coding in total? Please round to the nearest whole number, excluding any decimal points.', 'Did you spend time in the last year learning AI programming or AI-enabled tooling at work? Please answer with respect to your own field of work and professional experience.', 'Which of the following describes your current job or the job you had for the most time in the past year?', 'How many years of professional work experience do you have? Please round to the nearest whole number, excluding any decimal points.', 'Which programming, scripting, and markup languages have you done extensive development work in over the past year, and which do you want to work in over the next year?\xa0Please answer with respect to your own field of work and professional experience', 'Do you currently use AI tools in your development pro

## Bereinigung

In [14]:
# "N/A" steht für Fragen, die wegen Umfrageabbruch/Verzweigung nicht erreicht wurden -> NaN
# Kuriose Apostrophe (’) aus dem Survey-Export werden auf normale (') vereinheitlicht
df_clean = df_completed.copy()
for col in df_clean.select_dtypes(include=["object", "string"]).columns:
    df_clean[col] = df_clean[col].str.strip().str.replace("\u2019", "'", regex=False)
df_clean = df_clean.replace({"N/A": pd.NA, "": pd.NA})

df_clean.isna().sum().sort_values(ascending=False).head(20)

Which projects or operational constraints have the greatest influence on software development decisions in your field of work? Please briefly explain why the selected constraints have a particularly strong influence on software development decisions in your field of work. [I don't know][Comment]                                        26
Is there anything else you would like to share that is relevant to this topic?                                                                                                                                                                                                                                                                  26
Which projects or operational constraints have the greatest influence on software development decisions in your field of work? Please briefly explain why the selected constraints have a particularly strong influence on software development decisions in your field of work. [Limited opportunities for maintenance or updates

In [15]:
PROCESSED_PATH.parent.mkdir(parents=True, exist_ok=True)
df_clean.to_csv(PROCESSED_PATH, index=False)
print(f"Gespeichert unter {PROCESSED_PATH.resolve()}")

Gespeichert unter C:\Users\andre\OneDrive - Chemische Werke Kluthe GmbH\Dokumente\Vorlesungen 8tes Semester\Bachelorarbeit\Analysis Survey\data\processed\survey_clean.csv
